# Observer training end-to-end

Этот ноутбук проводит через полный pipeline обучения observer:
1. загружает и анализирует `teacher_encoded.json`,
2. собирает observer input manifests,
3. раскладывает encoded songs и строит teacher scalar targets,
4. валидирует соответствие manifests ↔ targets,
5. делает smoke checks dataset/graph/model,
6. запускает обучение observer,
7. показывает метрики,
8. выполняет sanity-check inference.


In [2]:
from __future__ import annotations

import json
import math
import random
import sys
from argparse import Namespace
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint
from typing import Any
from unittest.mock import patch

import numpy as np
import pandas as pd
import torch
from IPython.display import display
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataloader.theory_helpers import build_theory_context
from src.observer.build_teacher_targets import build_teacher_targets, load_jsonl_rows, write_jsonl
from src.observer.dataset import ObserverDataset
from src.observer.model import ObserverGNN
from src.observer.schema import OBSERVER_EDGE_TYPES, OBSERVER_NUM_FIELDS, build_observer_vocab_sizes
from src.observer.train_observer import create_loss, main as train_observer_main


/home/str/Fine-tune-text2midi-llm-with-gnn-theory-critic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ====== Central config ======
PROJECT_ROOT = Path(".").resolve()

TEACHER_ENCODED_PATH = PROJECT_ROOT / "data/HTCanon/encoded_full/teacher_encoded.json"
RENDERED_ROOT = PROJECT_ROOT / "data/rendered"
OUTPUT_ROOT = PROJECT_ROOT / "outputs/observer_training"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
TARGET_DIR = OUTPUT_ROOT / "targets"
TRAINING_DIR = OUTPUT_ROOT / "training_run"
ENCODED_SONGS_DIR = OUTPUT_ROOT / "encoded_songs"

TEACHER_CHECKPOINT = PROJECT_ROOT / "outputs/2026-04-14/08-39-46_full_data_teacher_gnn_base_pool-mean_max_hyb-True_bs-32/checkpoints/last.pt"
TEACHER_CONFIG = PROJECT_ROOT / "outputs/2026-04-14/08-39-46_full_data_teacher_gnn_base_pool-mean_max_hyb-True_bs-32/composed_config.yaml"

BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 0.0
HIDDEN_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
SEED = 42
LOSS = "smooth_l1"
IN_MEMORY = False
USE_FALLBACK_44 = True
CHORD_WEIGHTS_YAML = None
CHORD_INSTRUMENT_NAME = "chords"

LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

for p in [OUTPUT_ROOT, MANIFEST_DIR, TARGET_DIR, TRAINING_DIR, ENCODED_SONGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
assert TEACHER_ENCODED_PATH.exists(), f"Missing teacher encoded file: {TEACHER_ENCODED_PATH}"
assert TEACHER_CHECKPOINT.exists(), f"Missing teacher checkpoint: {TEACHER_CHECKPOINT}"
assert TEACHER_CONFIG.exists(), f"Missing teacher config: {TEACHER_CONFIG}"


## Helpers


In [ ]:
SPLITS = ("train", "val", "test")
LIMIT_BY_SPLIT = {"train": LIMIT_TRAIN, "val": LIMIT_VAL, "test": LIMIT_TEST}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl_local(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def preview_df(rows: list[dict[str, Any]], n: int = 5) -> pd.DataFrame:
    return pd.DataFrame(rows[:n]) if rows else pd.DataFrame()


## 3) Загрузка teacher_encoded.json и первичный обзор


In [ ]:
teacher_encoded = load_json(TEACHER_ENCODED_PATH)
assert isinstance(teacher_encoded, dict), "teacher_encoded root must be dict(song_id -> song_obj)"

total_tracks = len(teacher_encoded)
split_counter = Counter()
for song_id, song_obj in teacher_encoded.items():
    split = ((song_obj or {}).get("meta") or {}).get("split", "<missing>")
    split_counter[str(split)] += 1

print("Total tracks:", total_tracks)
print("Split counts:")
display(pd.DataFrame([{"split": k, "count": v} for k, v in sorted(split_counter.items())]))

example_song_id = next(iter(teacher_encoded))
example_song = teacher_encoded[example_song_id]
print("Example song_id:", example_song_id)
print("Example song object keys:", list(example_song.keys())[:20])
print("Example meta:")
pprint((example_song or {}).get("meta", {}))


## 4) Сбор observer manifests из teacher_encoded.json


In [ ]:
vocab_key_scale = load_json(PROJECT_ROOT / "metadata/vocabs/vocab_key_scale.json")
id_to_mode_name = {int(v): k for k, v in vocab_key_scale.items()}

missing_midi_rows: list[dict[str, Any]] = []
manifest_rows_by_split: dict[str, list[dict[str, Any]]] = {s: [] for s in SPLITS}
teacher_rows_by_split: dict[str, int] = {s: 0 for s in SPLITS}

for song_id, song_obj in teacher_encoded.items():
    meta = (song_obj or {}).get("meta") or {}
    split = str(meta.get("split", "")).strip()
    if split not in SPLITS:
        continue
    teacher_rows_by_split[split] += 1

    tonic_pc = meta.get("main_key_tonic_pc", meta.get("tonic_pc", None))
    scale_raw = meta.get("main_key_scale_id", meta.get("mode_name", None))
    if isinstance(scale_raw, str):
        mode_name = scale_raw
    elif scale_raw is None:
        mode_name = None
    else:
        mode_name = id_to_mode_name.get(int(scale_raw))

    midi_path = RENDERED_ROOT / split / f"{song_id}.mid"
    if not midi_path.exists():
        missing_midi_rows.append({"song_id": song_id, "split": split, "midi_path": str(midi_path)})
        continue

    row = {
        "song_id": str(song_id),
        "midi_path": str(midi_path),
        "tonic_pc": int(tonic_pc) if tonic_pc is not None else None,
        "mode_name": mode_name,
        "bpm": meta.get("main_bpm", None),
        "num_beats": meta.get("main_num_beats", None),
        "beat_unit": meta.get("main_beat_unit", None),
        "beat_origin": 1.0,
        "split": split,
    }
    manifest_rows_by_split[split].append(row)

for split in SPLITS:
    limit = LIMIT_BY_SPLIT[split]
    rows = manifest_rows_by_split[split]
    if limit is not None:
        rows = rows[: int(limit)]
    manifest_rows_by_split[split] = rows

manifest_paths = {
    "train": MANIFEST_DIR / "observer_train_input.jsonl",
    "val": MANIFEST_DIR / "observer_val_input.jsonl",
    "test": MANIFEST_DIR / "observer_test_input.jsonl",
}
for split, path in manifest_paths.items():
    write_jsonl_local(path, manifest_rows_by_split[split])

summary = []
for split in SPLITS:
    summary.append({
        "split": split,
        "teacher_rows": teacher_rows_by_split[split],
        "manifest_rows": len(manifest_rows_by_split[split]),
        "dropped_missing_midi": sum(1 for r in missing_midi_rows if r["split"] == split),
    })

display(pd.DataFrame(summary))
print(f"Total missing MIDI rows: {len(missing_midi_rows)}")
display(pd.DataFrame(missing_midi_rows[:10]))

for split in SPLITS:
    print(f"\nPreview {split} manifest")
    display(preview_df(manifest_rows_by_split[split], n=5))


> `beat_origin=1.0` установлен явно для совместимости с текущим teacher-style beat space observer pipeline.


## 5) Дополнительная валидация manifest'ов


In [ ]:
validation_errors: list[dict[str, Any]] = []

for split, rows in manifest_rows_by_split.items():
    if len(rows) == 0:
        validation_errors.append({"split": split, "song_id": None, "error": "empty split"})
        continue

    seen = set()
    for row in rows:
        song_id = row.get("song_id")
        midi_path = Path(str(row.get("midi_path", "")))
        tonic_pc = row.get("tonic_pc")
        mode_name = row.get("mode_name")
        bpm = row.get("bpm")
        num_beats = row.get("num_beats")
        beat_unit = row.get("beat_unit")

        if song_id in seen:
            validation_errors.append({"split": split, "song_id": song_id, "error": "duplicate song_id"})
        seen.add(song_id)

        if not midi_path.exists():
            validation_errors.append({"split": split, "song_id": song_id, "error": "missing midi_path"})
        if not isinstance(tonic_pc, int) or not (0 <= tonic_pc <= 11):
            validation_errors.append({"split": split, "song_id": song_id, "error": f"invalid tonic_pc={tonic_pc}"})
        if not isinstance(mode_name, str) or not mode_name.strip():
            validation_errors.append({"split": split, "song_id": song_id, "error": f"invalid mode_name={mode_name}"})
        if bpm is None or num_beats is None or beat_unit is None:
            validation_errors.append({"split": split, "song_id": song_id, "error": "missing bpm/num_beats/beat_unit"})

if validation_errors:
    print(f"Validation errors: {len(validation_errors)}")
    display(pd.DataFrame(validation_errors))
    raise ValueError("Manifest validation failed; fix the rows above.")
else:
    print("Manifest validation passed for all splits.")


## 6) Раскладка encoded songs и построение teacher targets


In [ ]:
# 6.1 unpack teacher_encoded dict -> per-song json files
encoded_song_counts = defaultdict(int)
for song_id, song_obj in teacher_encoded.items():
    split = str(((song_obj or {}).get("meta") or {}).get("split", "")).strip()
    if split not in SPLITS:
        continue
    if song_id not in {r["song_id"] for r in manifest_rows_by_split[split]}:
        continue
    out_path = ENCODED_SONGS_DIR / split / f"{song_id}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(song_obj, ensure_ascii=False), encoding="utf-8")
    encoded_song_counts[split] += 1

display(pd.DataFrame([{"split": s, "encoded_song_files": encoded_song_counts[s]} for s in SPLITS]))

# 6.2 build targets with src.observer.build_teacher_targets

target_paths = {
    "train": TARGET_DIR / "train_targets.jsonl",
    "val": TARGET_DIR / "val_targets.jsonl",
    "test": TARGET_DIR / "test_targets.jsonl",
}

target_rows_by_split: dict[str, list[dict[str, Any]]] = {}
for split in SPLITS:
    in_rows = load_jsonl_rows(manifest_paths[split])
    out_rows = build_teacher_targets(
        rows=in_rows,
        teacher_checkpoint=TEACHER_CHECKPOINT,
        teacher_config=TEACHER_CONFIG,
        encoded_song_field="encoded_song_path",
        encoded_song_root=ENCODED_SONGS_DIR,
        split=split,
        device=DEVICE,
    )
    write_jsonl(target_paths[split], out_rows)
    target_rows_by_split[split] = out_rows

for split in SPLITS:
    rows = target_rows_by_split[split]
    scores = np.asarray([float(r["teacher_score"]) for r in rows], dtype=float) if rows else np.asarray([], dtype=float)
    print(f"\nSplit={split}: n={len(rows)}")
    display(preview_df(rows, n=5))
    if scores.size:
        print({
            "min": float(np.min(scores)),
            "max": float(np.max(scores)),
            "mean": float(np.mean(scores)),
            "std": float(np.std(scores)),
            "nan_count": int(np.isnan(scores).sum()),
            "inf_count": int(np.isinf(scores).sum()),
        })


## 7) Проверка соответствия manifest ↔ targets


In [ ]:
rows = []
for split in SPLITS:
    m_rows = read_jsonl(manifest_paths[split])
    t_rows = read_jsonl(target_paths[split])
    m_ids = {r["song_id"] for r in m_rows}
    t_ids = {r["song_id"] for r in t_rows}
    missing_targets = sorted(m_ids - t_ids)
    extra_targets = sorted(t_ids - m_ids)
    ok = (len(m_ids) == len(t_ids)) and (not missing_targets) and (not extra_targets)
    rows.append({
        "split": split,
        "manifest_n": len(m_rows),
        "targets_n": len(t_rows),
        "missing_targets_n": len(missing_targets),
        "extra_targets_n": len(extra_targets),
        "ok": ok,
    })
    if missing_targets:
        print(f"{split} missing targets (first 10):", missing_targets[:10])
    if extra_targets:
        print(f"{split} extra targets (first 10):", extra_targets[:10])

df_match = pd.DataFrame(rows)
display(df_match)
assert df_match["ok"].all(), "Manifest/targets mismatch detected"


## 8) Smoke check ObserverDataset


In [ ]:
set_seed(SEED)

dataset_kwargs = {
    "chord_weights_yaml": CHORD_WEIGHTS_YAML,
    "chord_instrument_name": CHORD_INSTRUMENT_NAME,
    "use_fallback_44": USE_FALLBACK_44,
    "in_memory": IN_MEMORY,
}

train_dataset = ObserverDataset(manifest_paths["train"], target_paths["train"], **dataset_kwargs)
val_dataset = ObserverDataset(manifest_paths["val"], target_paths["val"], **dataset_kwargs)

assert len(train_dataset) > 0, "train dataset is empty"
assert len(val_dataset) > 0, "val dataset is empty"
print("train len:", len(train_dataset), "val len:", len(val_dataset))

graph0 = train_dataset[0]
assert hasattr(graph0, "y") and torch.isfinite(graph0.y).all(), "graph.y missing or non-finite"
assert hasattr(graph0, "song_id"), "graph.song_id missing"

node_shapes = []
for nt in graph0.node_types:
    store = graph0[nt]
    node_shapes.append({
        "node_type": nt,
        "num_nodes": int(store.num_nodes),
        "x_cat_shape": tuple(store.x_cat.shape),
        "x_num_shape": tuple(store.x_num.shape),
        "x_shape": tuple(store.x.shape),
    })

edge_types = ["__".join(et) for et in graph0.edge_types]

print("song_id:", graph0.song_id)
print("graph.y:", float(graph0.y.item()))
display(pd.DataFrame(node_shapes))
print("edge types:")
for et in edge_types:
    print(" -", et)


## 9) Mini-batch smoke check


In [ ]:
mini_loader = DataLoader(train_dataset, batch_size=min(BATCH_SIZE, 2), shuffle=False, num_workers=0)
mini_batch = next(iter(mini_loader)).to(torch.device(DEVICE))

spec_global = load_json(PROJECT_ROOT / "metadata/specs/spec_global.json")
model = ObserverGNN(
    cat_vocab_sizes=build_observer_vocab_sizes(build_theory_context(), spec_global),
    num_feature_dims={node_type: len(OBSERVER_NUM_FIELDS[node_type]) for node_type in OBSERVER_NUM_FIELDS},
    edge_types=OBSERVER_EDGE_TYPES,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(torch.device(DEVICE))

preds = model(mini_batch).view(-1)
targets = mini_batch.y.view(-1).float()
criterion = create_loss(LOSS)
loss = criterion(preds, targets)

print("batch_size:", int(targets.shape[0]))
print("preds:", preds.detach().cpu().numpy())
print("targets:", targets.detach().cpu().numpy())
print("loss:", float(loss.detach().cpu().item()))
assert preds.shape == targets.shape, "Prediction shape mismatch"
assert torch.isfinite(loss).item(), "Loss is not finite"


## 10) Запуск обучения observer


In [ ]:
train_args = Namespace(
    train_input_jsonl=manifest_paths["train"],
    train_target_jsonl=target_paths["train"],
    val_input_jsonl=manifest_paths["val"],
    val_target_jsonl=target_paths["val"],
    output_dir=TRAINING_DIR,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    device=DEVICE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    loss=LOSS,
    in_memory=IN_MEMORY,
    chord_weights_yaml=CHORD_WEIGHTS_YAML,
    chord_instrument_name=CHORD_INSTRUMENT_NAME,
    use_fallback_44=USE_FALLBACK_44,
)

print("Training config:")
display(pd.DataFrame([vars(train_args)]).T.rename(columns={0: "value"}))

for artifact in ["config.json", "metrics.jsonl", "last.pt", "best.pt"]:
    p = TRAINING_DIR / artifact
    if p.exists():
        p.unlink()

with patch("src.observer.train_observer.parse_args", return_value=train_args):
    train_observer_main()

for artifact in ["config.json", "metrics.jsonl", "last.pt", "best.pt"]:
    p = TRAINING_DIR / artifact
    print(artifact, "exists=", p.exists(), "path=", p)
    assert p.exists(), f"Missing training artifact: {artifact}"


## 11) Просмотр результатов обучения


In [ ]:
metrics_rows = read_jsonl(TRAINING_DIR / "metrics.jsonl")
assert metrics_rows, "metrics.jsonl is empty"

flat_rows = []
for r in metrics_rows:
    flat_rows.append({
        "epoch": r["epoch"],
        "train_loss": r["train"]["loss"],
        "train_mae": r["train"]["mae"],
        "train_rmse": r["train"]["rmse"],
        "train_pearson": r["train"].get("pearson"),
        "train_spearman": r["train"].get("spearman"),
        "val_loss": r["val"]["loss"],
        "val_mae": r["val"]["mae"],
        "val_rmse": r["val"]["rmse"],
        "val_pearson": r["val"].get("pearson"),
        "val_spearman": r["val"].get("spearman"),
    })

metrics_df = pd.DataFrame(flat_rows)
display(metrics_df)

last_epoch_row = metrics_df.iloc[-1]
best_val_idx = metrics_df["val_loss"].idxmin()
best_val_row = metrics_df.loc[best_val_idx]

print("Last epoch summary:")
display(last_epoch_row.to_frame(name="value"))
print("Best val loss:", float(best_val_row["val_loss"]), "at epoch", int(best_val_row["epoch"]))

plt.figure(figsize=(8, 4))
plt.plot(metrics_df["epoch"], metrics_df["train_loss"], marker="o", label="train_loss")
plt.plot(metrics_df["epoch"], metrics_df["val_loss"], marker="o", label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Train/Val Loss")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(metrics_df["epoch"], metrics_df["train_mae"], marker="o", label="train_mae")
plt.plot(metrics_df["epoch"], metrics_df["val_mae"], marker="o", label="val_mae")
plt.xlabel("epoch")
plt.ylabel("mae")
plt.title("Train/Val MAE")
plt.grid(True)
plt.legend()
plt.show()

if "val_pearson" in metrics_df and metrics_df["val_pearson"].notna().any():
    plt.figure(figsize=(8, 4))
    plt.plot(metrics_df["epoch"], metrics_df["train_pearson"], marker="o", label="train_pearson")
    plt.plot(metrics_df["epoch"], metrics_df["val_pearson"], marker="o", label="val_pearson")
    plt.plot(metrics_df["epoch"], metrics_df["train_spearman"], marker="o", label="train_spearman")
    plt.plot(metrics_df["epoch"], metrics_df["val_spearman"], marker="o", label="val_spearman")
    plt.xlabel("epoch")
    plt.ylabel("correlation")
    plt.title("Pearson/Spearman by epoch")
    plt.grid(True)
    plt.legend()
    plt.show()


## 12) Sanity-check inference на обученном observer


In [ ]:
device_t = torch.device(DEVICE)
ckpt = torch.load(TRAINING_DIR / "best.pt", map_location=device_t)
train_cfg = ckpt.get("config", {})

spec_global = load_json(PROJECT_ROOT / "metadata/specs/spec_global.json")
infer_model = ObserverGNN(
    cat_vocab_sizes=build_observer_vocab_sizes(build_theory_context(), spec_global),
    num_feature_dims={node_type: len(OBSERVER_NUM_FIELDS[node_type]) for node_type in OBSERVER_NUM_FIELDS},
    edge_types=OBSERVER_EDGE_TYPES,
    hidden_dim=int(train_cfg.get("hidden_dim", HIDDEN_DIM)),
    num_layers=int(train_cfg.get("num_layers", NUM_LAYERS)),
    dropout=float(train_cfg.get("dropout", DROPOUT)),
).to(device_t)
infer_model.load_state_dict(ckpt["model_state_dict"])
infer_model.eval()

sanity_split = "val" if len(val_dataset) > 0 else "test"
sanity_dataset = val_dataset if sanity_split == "val" else ObserverDataset(manifest_paths["test"], target_paths["test"], **dataset_kwargs)
subset_n = min(10, len(sanity_dataset))
subset_graphs = [sanity_dataset[i] for i in range(subset_n)]
subset_loader = DataLoader(subset_graphs, batch_size=min(4, subset_n), shuffle=False)

rows = []
with torch.no_grad():
    for batch in subset_loader:
        batch = batch.to(device_t)
        pred = infer_model(batch).view(-1).detach().cpu().numpy()
        tgt = batch.y.view(-1).detach().cpu().numpy()
        sid = list(batch.song_id)
        for song_id, p, t in zip(sid, pred, tgt):
            rows.append({
                "song_id": song_id,
                "teacher_score": float(t),
                "observer_score": float(p),
                "abs_error": float(abs(p - t)),
            })

sanity_df = pd.DataFrame(rows)
display(sanity_df)

if len(sanity_df) >= 2:
    pearson = sanity_df["teacher_score"].corr(sanity_df["observer_score"], method="pearson")
    spearman = sanity_df["teacher_score"].corr(sanity_df["observer_score"], method="spearman")
    print({"pearson": float(pearson), "spearman": float(spearman)})


## Next steps

- Запустить обучение на полном наборе (`LIMIT_* = None`, больше `EPOCHS`).
- Добавить отдельную финальную оценку на test split.
- Сравнить observer на corruption pairs / ranking-aware сценариях.
- Протестировать ranking loss или pairwise distillation для повышения согласованности порядка.
